In [20]:
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, HDBSCAN
# GaussianMixture
from sklearn.mixture import GaussianMixture

import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns
# import
from hmmlearn.hmm import GaussianHMM
import pickle

In [21]:
device ="mps"

In [22]:
with open('../../data/complete_features.pkl', 'rb') as f:
    all_data = pickle.load(f)

In [23]:
data = all_data["scaled_featured"].copy()

In [24]:
data

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20
Date,,,,,,,,,,,,
2008-07-07,-1.220376,-1.882225,2.081321,-0.864592,-1.135796,-0.628471,-0.569023,-2.119483,1.416877,0.823997,-2.165333,-0.568676
2008-07-08,-1.211754,-1.882225,2.077736,-0.022097,-1.117078,-1.046961,-0.542321,-2.455523,1.450259,0.981827,-2.206775,-0.513460
2008-07-09,-0.957426,-1.882225,2.289506,1.636228,-1.218247,-0.338846,-0.561373,-1.988903,1.476862,0.785786,-1.960992,-0.565134
2008-07-10,-0.975703,-1.882225,2.287995,-0.006124,-1.013921,-0.474801,-0.596656,-2.401313,1.493754,0.505013,-1.940271,-0.562557
2008-07-11,-1.136144,-1.882225,2.391761,0.804554,-1.050675,-0.396400,-0.475448,-2.987661,1.507027,0.400347,-2.078101,-0.510752
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15,0.375855,0.654743,-0.795604,0.021503,0.541634,-0.126723,-0.727148,-1.189892,-0.988672,-0.309058,0.314003,0.507783
2025-09-16,0.280399,0.654743,-0.833663,-0.287568,0.045456,-0.238463,-0.708679,-1.276291,-1.000565,-0.327333,0.453848,0.513354
2025-09-17,0.260114,0.654743,-0.836851,-0.019044,0.154682,-0.363408,-0.703390,-1.335486,-1.009414,-0.239281,0.522324,0.511624


In [25]:
close = all_data["Close"].copy()
close

Date
2008-07-07     4030.000000
2008-07-08     3988.550049
2008-07-09     4157.100098
2008-07-10     4162.200195
2008-07-11     4049.000000
                  ...     
2025-09-15    25069.199219
2025-09-16    25239.099609
2025-09-17    25330.250000
2025-09-18    25423.599609
2025-09-19    25327.050781
Name: Close, Length: 4219, dtype: float64

In [26]:
data.shape

(4219, 12)

In [27]:
class TimeSeriesWindowDataset(Dataset):
    def __init__(self, data, window_size=60):
        """
        data: numpy array [T, D]
        """
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.window_size]
        return x

window_size = 60
val_size = 0.2

train_size = int(len(data) * (1 - val_size))
train_data = data.iloc[:train_size]
val_data = data.iloc[train_size:]

train_dataset = TimeSeriesWindowDataset(train_data.values, window_size)
val_dataset = TimeSeriesWindowDataset(val_data.values, window_size)
combined_dataset = TimeSeriesWindowDataset(data.values, window_size)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
combined_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=False)



In [28]:
len(combined_dataset)

4160

In [29]:
for batch in train_loader:
    print(batch.shape)  # Should print torch.Size([32, 60, D])
    break

torch.Size([32, 60, 12])


In [30]:
import torch.nn as nn
import torch.nn.functional as F

class TS2VecModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=5):
        super(TS2VecModel, self).__init__()
        layers = []
        for i in range(num_layers):
            dilation = 2 ** i
            layers.append(nn.Conv1d(input_dim if i == 0 else hidden_dim, hidden_dim, kernel_size=3, padding=dilation, dilation=dilation))
            layers.append(nn.ReLU())
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # x: [B, T, D]
        x = x.permute(0, 2, 1)  # [B, D, T]
        x = self.network(x)     # [B, H, T]
        x = x.permute(0, 2, 1)  # [B, T, H]
        # normalize
        x = F.normalize(x, p=2, dim=-1)
        return x

In [31]:
data.shape

(4219, 12)

In [32]:
model = TS2VecModel(input_dim=data.shape[1], hidden_dim=128, num_layers=6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [33]:
model

TS2VecModel(
  (network): Sequential(
    (0): Conv1d(12, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
    (3): ReLU()
    (4): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(4,))
    (5): ReLU()
    (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(8,), dilation=(8,))
    (7): ReLU()
    (8): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(16,), dilation=(16,))
    (9): ReLU()
    (10): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(32,), dilation=(32,))
    (11): ReLU()
  )
)

In [34]:
def time_mask(x, mask_ratio=0.2):
    B, T, D = x.shape
    mask_len = int(T * mask_ratio)
    start = np.random.randint(0, T - mask_len)
    x = x.clone()
    x[:, start:start+mask_len, :] = 0
    return x


def jitter(x, sigma=0.02):
    return x + sigma * torch.randn_like(x)


def temporal_pooling(z):
    if z.size(1) % 2 == 1:
        z = z[:, :-1]
    return z.reshape(z.size(0), z.size(1)//2, 2, z.size(2)).mean(dim=2)


def ts2vec_contrastive_loss_vectorized(
    z1, z2,
    temperature=0.1,
    base_exclusion_radius=5
):
    """
    z1, z2: [B, T, C]
    """
    B, T, C = z1.shape
    device = z1.device

    # flatten (instance, time)
    z1_flat = z1.reshape(B*T, C)
    z2_flat = z2.reshape(B*T, C)

    # cosine similarity == dot product because embeddings are normalized
    sim = torch.matmul(z1_flat, z2_flat.T) / temperature   # [BT, BT]

    # ----- temporal negative mask -----
    exclusion_radius = min(base_exclusion_radius, (T - 1) // 2)

    if exclusion_radius == 0:
        return torch.tensor(0.0, device=device)

    # time index per row
    time_idx = torch.arange(T, device=device).repeat(B)     # [BT]

    # batch index per row
    batch_idx = torch.arange(B, device=device).repeat_interleave(T)

    # same batch & temporally close → mask out
    temporal_dist = torch.abs(time_idx[:, None] - time_idx[None, :])
    same_batch = batch_idx[:, None] == batch_idx[None, :]

    invalid_negatives = same_batch & (temporal_dist <= exclusion_radius)

    # allow diagonal (positive pairs)
    diag = torch.eye(B*T, device=device, dtype=torch.bool)
    invalid_negatives = invalid_negatives & (~diag)

    # mask invalid negatives
    sim = sim.masked_fill(invalid_negatives, -1e9)

    # positives are diagonal
    labels = torch.arange(B*T, device=device)

    return F.cross_entropy(sim, labels)



def hierarchical_ts2vec_loss_v2(
    z1, z2,
    temperature=0.1,
    exclusion_radius=5,
    min_time=2
):
    total_loss = 0.0
    depth = 0

    while z1.size(1) >= min_time:
        T = z1.size(1)

        if T > 2 * exclusion_radius + 1:
            total_loss += ts2vec_contrastive_loss_vectorized(
                z1, z2,
                temperature,
                exclusion_radius
            )
            depth += 1

        z1 = temporal_pooling(z1)
        z2 = temporal_pooling(z2)

    return total_loss / max(depth, 1)



In [36]:
%%time
for epoch in range(3):
    total_loss = 0
    model.train()
    for x in train_loader:
        x=x.to(device)
        x1 = jitter(time_mask(x))
        x2 = jitter(time_mask(x))
        z1 = model(x1)
        z2 = model(x2)


        loss = hierarchical_ts2vec_loss_v2(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        # print(f"Batch Loss: {loss.item():.4f}")
    # model.eval()
    # with torch.no_grad():
    #     val_loss = 0
    #     for x in val_loader:
    #         x = x.to(device)
    #         x1 = jitter(time_mask(x))
    #         x2 = jitter(time_mask(x))
    #         z1 = model(x1)
    #         z2 = model(x2)
    #         val_loss += hierarchical_ts2vec_loss_v2(z1, z2).item()

    # val_loss /= len(val_loader)

    print(f"Epoch {epoch}, Loss {total_loss / len(train_loader):.4f}")

Epoch 0, Loss 1.6901
Epoch 1, Loss 0.6712
Epoch 2, Loss 0.5300
CPU times: user 2.67 s, sys: 571 ms, total: 3.24 s
Wall time: 2.77 s


In [159]:
# save model
# torch.save(model.state_dict(), './../../models/ts2vec_nifty.pth')

In [35]:
# load model
# model.load_state_dict(torch.load('../../models/ts2vec_nifty.pth'))

In [37]:
# use full dataset to get embeddings without dataloader
model.eval()
with torch.no_grad():
    full_data_tensor = torch.tensor(data.values, dtype=torch.float32).unsqueeze(0).to(device)  # [1, T, D]
    z=model(full_data_tensor).squeeze(0).cpu().numpy()

In [41]:
temporal_contrast_score_multi(z)

0.68542373

In [161]:
all_embeddings = []
with torch.inference_mode():
    for x in combined_loader:
        # print("x.shape", x.shape)
        x = x.to(device)    
        z = model(x)          # [B, T, C]
        z_mean = z.mean(dim=1)  # [B, C]
        all_embeddings.append(z_mean)

embeddings_full = torch.cat(all_embeddings).cpu().numpy()

In [162]:
len(embeddings_full), len(data) # 4219

(4160, 4219)

In [163]:
val_df = pd.DataFrame(
    embeddings_full,
    index=data.index[window_size - 1:]
)

In [164]:
# val_df["Close"] = all_data["Close"].iloc[window_size-1:].values

In [165]:
val_df.head()

,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
Date,,,,,,,,,,,,,,,,,,,,,
2008-09-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000035,0.0,0.000000,...,0.215032,0.000000,0.0,0.0,0.104896,0.0,0.0,0.071229,0.0,0.040119
2008-10-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000174,0.0,0.000000,...,0.206248,0.000000,0.0,0.0,0.104896,0.0,0.0,0.073362,0.0,0.044781
2008-10-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000233,0.0,0.000124,...,0.198753,0.001280,0.0,0.0,0.104896,0.0,0.0,0.075323,0.0,0.049510
2008-10-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000672,0.0,0.000035,...,0.191434,0.003058,0.0,0.0,0.104896,0.0,0.0,0.076693,0.0,0.056227
2008-10-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000484,0.0,0.000000,...,0.183625,0.004119,0.0,0.0,0.104896,0.0,0.0,0.076525,0.0,0.062015


In [40]:
def temporal_contrast_score(Z, k=20):
    """
    This function computes the temporal contrast score for a given set of embeddings Z.
    Z: [T, C] numpy array of embeddings
    k: temporal gap for negative pairs
    """
    pos = []
    neg = []
    
    for i in range(len(Z) - k - 1):
        pos.append(np.dot(Z[i], Z[i+1]))
        neg.append(np.dot(Z[i], Z[i+k]))
    
    return np.mean(pos) - np.mean(neg)

# score = temporal_contrast_score(embeddings_full)
def temporal_contrast_score_multi(Z, pos_gap=1, neg_gaps=(10,20,40)):
    pos = []
    neg = []

    for i in range(len(Z) - max(neg_gaps) - 1):
        pos.append(np.dot(Z[i], Z[i+pos_gap]))
        for k in neg_gaps:
            neg.append(np.dot(Z[i], Z[i+k]))

    return np.mean(pos) - np.mean(neg)

In [142]:
score

0.077012636

In [17]:
from dataclasses import dataclass

@dataclass
class WindowParams:
    window_size: int

@dataclass
class ArchitectureParams:
    hidden_dim: int
    num_layers: int

@dataclass
class ContrastiveParams:
    temperature: float
    exclusion_radius: int

@dataclass
class AugmentationParams:
    mask_ratio: float
    jitter_sigma: float

@dataclass
class OptimizationParams:
    learning_rate: float
    batch_size: int

In [53]:
import optuna
import torch
import numpy as np

class BaseTS2VecObjective:
    def __init__(self, data, device, fixed_params):
        self.data = data
        self.device = device
        self.fixed_params = fixed_params  # dict of params from previous stages

    def build_model(self, arch_params):
        return TS2VecModel(
            input_dim=self.data.shape[1],
            hidden_dim=arch_params.hidden_dim,
            num_layers=arch_params.num_layers
        ).to(self.device)

    def compute_score(self, model):
        model.eval()
        with torch.no_grad():
            full_tensor = torch.tensor(self.data.values, dtype=torch.float32).unsqueeze(0).to(self.device)
            z = model(full_tensor).squeeze(0).cpu().numpy()
            return temporal_contrast_score_multi(z)

    def train_model(self, model, train_loader, contrastive_params, aug_params, opt_params):
        optimizer = torch.optim.Adam(model.parameters(), lr=opt_params.learning_rate)

        model.train()
        for _ in range(3):  # small epochs for HPO
            for x in train_loader:
                x = x.to(self.device)
                x1 = jitter(time_mask(x, aug_params.mask_ratio), aug_params.jitter_sigma)
                x2 = jitter(time_mask(x, aug_params.mask_ratio), aug_params.jitter_sigma)

                z1 = model(x1)
                z2 = model(x2)

                loss = hierarchical_ts2vec_loss_v2(
                    z1, z2,
                    temperature=contrastive_params.temperature,
                    exclusion_radius=contrastive_params.exclusion_radius
                )

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

In [54]:
def create_loader(data, window_size, batch_size):
    dataset = TimeSeriesWindowDataset(data.values, window_size)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


In [58]:

class WindowStageObjective(BaseTS2VecObjective):

    def __call__(self, trial):
        window_size = trial.suggest_int("window_size", 30, 180, step=30)

        window_params = WindowParams(window_size)

        train_loader = create_loader(self.data, window_size, batch_size=64)

        # fixed architecture for stage 0
        arch_params = ArchitectureParams(hidden_dim=128, num_layers=5)
        contrastive_params = ContrastiveParams(0.1, 5)
        aug_params = AugmentationParams(0.2, 0.02)
        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("window_params", window_params.__dict__)

        return score

In [56]:
type(data.values)

numpy.ndarray

In [59]:
study_window = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="window_stage",
    load_if_exists=True
)

# suggest optimal trail count based on combination of parameters 


study_window.optimize(WindowStageObjective(data, device, {}), n_trials=7)

[I 2026-02-22 13:31:11,194] A new study created in RDB with name: window_stage
[I 2026-02-22 13:31:18,083] Trial 0 finished with value: 0.7937400937080383 and parameters: {'window_size': 120}. Best is trial 0 with value: 0.7937400937080383.
[I 2026-02-22 13:31:20,125] Trial 1 finished with value: 0.7081829309463501 and parameters: {'window_size': 60}. Best is trial 0 with value: 0.7937400937080383.
[I 2026-02-22 13:31:26,900] Trial 2 finished with value: 0.7790645360946655 and parameters: {'window_size': 120}. Best is trial 0 with value: 0.7937400937080383.
[I 2026-02-22 13:31:41,269] Trial 3 finished with value: 0.8101174235343933 and parameters: {'window_size': 180}. Best is trial 3 with value: 0.8101174235343933.
[I 2026-02-22 13:31:48,103] Trial 4 finished with value: 0.7787735462188721 and parameters: {'window_size': 120}. Best is trial 3 with value: 0.8101174235343933.
[I 2026-02-22 13:31:50,187] Trial 5 finished with value: 0.7155215740203857 and parameters: {'window_size': 60}.

In [60]:
class ArchitectureStageObjective(BaseTS2VecObjective):

    def __call__(self, trial):

        hidden_dim = trial.suggest_categorical("hidden_dim", [64, 128, 256])
        num_layers = trial.suggest_int("num_layers", 3, 7)

        arch_params = ArchitectureParams(hidden_dim, num_layers)

        window_size = self.fixed_params["window_size"]

        train_loader = create_loader(self.data, window_size, batch_size=64)

        contrastive_params = ContrastiveParams(0.1, 5)
        aug_params = AugmentationParams(0.2, 0.02)
        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("arch_params", arch_params.__dict__)

        return score
    
study_arch = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="arch_stage",
    load_if_exists=True
)

# suggest n_trials count based on number of combinations (3 hidden_dim options * 5 num_layers options = 15 combinations => suggest 30 trials to cover each combination at least twice)
n_trials = 30

study_arch.optimize(ArchitectureStageObjective(data, device, study_window.best_trial.user_attrs["window_params"]), n_trials= 30)

[I 2026-02-22 13:32:15,033] A new study created in RDB with name: arch_stage
[I 2026-02-22 13:32:34,677] Trial 0 finished with value: 0.7654034495353699 and parameters: {'hidden_dim': 256, 'num_layers': 7}. Best is trial 0 with value: 0.7654034495353699.
[I 2026-02-22 13:32:54,014] Trial 1 finished with value: 0.781647801399231 and parameters: {'hidden_dim': 256, 'num_layers': 7}. Best is trial 1 with value: 0.781647801399231.
[I 2026-02-22 13:33:08,328] Trial 2 finished with value: 0.7660345435142517 and parameters: {'hidden_dim': 128, 'num_layers': 3}. Best is trial 1 with value: 0.781647801399231.
[I 2026-02-22 13:33:22,557] Trial 3 finished with value: 0.737932562828064 and parameters: {'hidden_dim': 128, 'num_layers': 3}. Best is trial 1 with value: 0.781647801399231.
[I 2026-02-22 13:33:35,746] Trial 4 finished with value: 0.7127401828765869 and parameters: {'hidden_dim': 64, 'num_layers': 4}. Best is trial 1 with value: 0.781647801399231.
[I 2026-02-22 13:33:50,204] Trial 5 fini

In [66]:
# Contrastive stage:

# temperature = trial.suggest_float("temperature", 0.05, 0.5, log=True)
# exclusion_radius = trial.suggest_int("exclusion_radius", 2, 20)

class ContrastiveStageObjective(BaseTS2VecObjective):

    def __call__(self, trial):

        temperature = trial.suggest_float("temperature", 0.05, 0.5, log=True)
        exclusion_radius = trial.suggest_int("exclusion_radius", 2, 20)

        contrastive_params = ContrastiveParams(temperature, exclusion_radius)

        window_size = self.fixed_params["window_size"]
        arch_params = ArchitectureParams(self.fixed_params["hidden_dim"], self.fixed_params["num_layers"])

        train_loader = create_loader(self.data, window_size, batch_size=64)

        aug_params = AugmentationParams(0.2, 0.02)
        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("contrastive_params", contrastive_params.__dict__)

        return score
    
study_contrastive = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="contrastive_stage",
    load_if_exists=True
)

study_contrastive.optimize(ContrastiveStageObjective(data, device, 
                                                     {
    **study_window.best_trial.user_attrs["window_params"],
    **study_arch.best_trial.user_attrs["arch_params"]
}
), n_trials=50)

[I 2026-02-22 14:36:38,179] Using an existing study with name 'contrastive_stage' instead of creating a new one.


[I 2026-02-22 14:36:55,646] Trial 2 finished with value: 0.7239099740982056 and parameters: {'temperature': 0.20124654648298504, 'exclusion_radius': 9}. Best is trial 2 with value: 0.7239099740982056.
[I 2026-02-22 14:37:13,097] Trial 3 finished with value: 0.7523349523544312 and parameters: {'temperature': 0.06868513558308588, 'exclusion_radius': 4}. Best is trial 3 with value: 0.7523349523544312.
[I 2026-02-22 14:37:30,429] Trial 4 finished with value: 0.5766209959983826 and parameters: {'temperature': 0.0535096474830974, 'exclusion_radius': 5}. Best is trial 3 with value: 0.7523349523544312.
[I 2026-02-22 14:37:47,662] Trial 5 finished with value: 0.8162449598312378 and parameters: {'temperature': 0.15466840894620842, 'exclusion_radius': 13}. Best is trial 5 with value: 0.8162449598312378.
[I 2026-02-22 14:38:04,924] Trial 6 finished with value: 0.7584587335586548 and parameters: {'temperature': 0.17928329679413899, 'exclusion_radius': 16}. Best is trial 5 with value: 0.816244959831

In [67]:
# Augmentation stage:

# mask_ratio = trial.suggest_float("mask_ratio", 0.1, 0.4)
# jitter_sigma = trial.suggest_float("jitter_sigma", 0.005, 0.05, log=True)

class AugmentationStageObjective(BaseTS2VecObjective):
    
    def __call__(self, trial):

        mask_ratio = trial.suggest_float("mask_ratio", 0.1, 0.4)
        jitter_sigma = trial.suggest_float("jitter_sigma", 0.005, 0.05, log=True)

        aug_params = AugmentationParams(mask_ratio, jitter_sigma)

        window_size = self.fixed_params["window_size"]
        arch_params = ArchitectureParams(self.fixed_params["hidden_dim"], self.fixed_params["num_layers"])
        contrastive_params = ContrastiveParams(self.fixed_params["temperature"], self.fixed_params["exclusion_radius"])

        train_loader = create_loader(self.data, window_size, batch_size=64)

        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("aug_params", aug_params.__dict__)

        return score
    
study_aug = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="augmentation_stage",
    load_if_exists=True
)
study_aug.optimize(AugmentationStageObjective(data, device, 
                                                     {
    **study_window.best_trial.user_attrs["window_params"],
    **study_arch.best_trial.user_attrs["arch_params"],
    **study_contrastive.best_trial.user_attrs["contrastive_params"]
}
), n_trials=50)

[I 2026-02-22 14:55:14,501] A new study created in RDB with name: augmentation_stage
[I 2026-02-22 14:55:31,973] Trial 0 finished with value: 0.822404146194458 and parameters: {'mask_ratio': 0.3160850114434691, 'jitter_sigma': 0.0067355966719553914}. Best is trial 0 with value: 0.822404146194458.
[I 2026-02-22 14:55:49,260] Trial 1 finished with value: 0.8854341506958008 and parameters: {'mask_ratio': 0.12468497408563146, 'jitter_sigma': 0.005892071487060923}. Best is trial 1 with value: 0.8854341506958008.
[I 2026-02-22 14:56:06,589] Trial 2 finished with value: 0.7065039873123169 and parameters: {'mask_ratio': 0.3534629492038706, 'jitter_sigma': 0.010486868782593584}. Best is trial 1 with value: 0.8854341506958008.
[I 2026-02-22 14:56:23,992] Trial 3 finished with value: 0.8739010095596313 and parameters: {'mask_ratio': 0.1607389580190983, 'jitter_sigma': 0.040570314897763644}. Best is trial 1 with value: 0.8854341506958008.
[I 2026-02-22 14:56:41,518] Trial 4 finished with value: 0.

In [68]:
# Optimization stage:

# learning_rate = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)
# batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

class OptimizationStageObjective(BaseTS2VecObjective):
    
    def __call__(self, trial):

        learning_rate = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

        opt_params = OptimizationParams(learning_rate, batch_size)

        window_size = self.fixed_params["window_size"]
        arch_params = ArchitectureParams(self.fixed_params["hidden_dim"], self.fixed_params["num_layers"])
        contrastive_params = ContrastiveParams(self.fixed_params["temperature"], self.fixed_params["exclusion_radius"])
        aug_params = AugmentationParams(self.fixed_params["mask_ratio"], self.fixed_params["jitter_sigma"])

        train_loader = create_loader(self.data, window_size, batch_size)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("opt_params", opt_params.__dict__)

        return score
    
study_opt = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="optimization_stage",
    load_if_exists=True
)
study_opt.optimize(OptimizationStageObjective(data, device,
                                                     {
    **study_window.best_trial.user_attrs["window_params"],
    **study_arch.best_trial.user_attrs["arch_params"],
    **study_contrastive.best_trial.user_attrs["contrastive_params"],
    **study_aug.best_trial.user_attrs["aug_params"]
}), n_trials=50)

[I 2026-02-22 15:20:45,762] A new study created in RDB with name: optimization_stage
[I 2026-02-22 15:20:55,699] Trial 0 finished with value: 0.8872004747390747 and parameters: {'learning_rate': 0.001964145474122741, 'batch_size': 32}. Best is trial 0 with value: 0.8872004747390747.
[I 2026-02-22 15:21:29,934] Trial 1 finished with value: 0.8732810020446777 and parameters: {'learning_rate': 0.0026197575781272244, 'batch_size': 128}. Best is trial 0 with value: 0.8872004747390747.
[I 2026-02-22 15:21:47,272] Trial 2 finished with value: 0.89781653881073 and parameters: {'learning_rate': 0.0009812743476000573, 'batch_size': 64}. Best is trial 2 with value: 0.89781653881073.
[I 2026-02-22 15:22:21,415] Trial 3 finished with value: 0.8366058468818665 and parameters: {'learning_rate': 0.0002696060521162218, 'batch_size': 128}. Best is trial 2 with value: 0.89781653881073.
[I 2026-02-22 15:22:55,188] Trial 4 finished with value: 0.8967544436454773 and parameters: {'learning_rate': 0.00161462

In [ ]:
class TS2VecHyperparameterPipeline:

    def __init__(self, data, device):
        self.data = data
        self.device = device
        self.best_params = {}

    def run_stage(self, objective_class, n_trials=30):
        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler()
        )

        study.optimize(
            objective_class(self.data, self.device, self.best_params),
            n_trials=n_trials
        )

        self.best_params.update(study.best_params)

        return study

    def run_full_pipeline(self):
        study_window = self.run_stage(WindowStageObjective)
        study_arch = self.run_stage(ArchitectureStageObjective)
        # Add remaining stages here
        study_contrastive = self.run_stage(ContrastiveStageObjective)
        study_aug = self.run_stage(AugmentationStageObjective)
        study_opt = self.run_stage(OptimizationStageObjective)
        return {
            "window": study_window,
            "architecture": study_arch,
            "contrastive": study_contrastive,
            "augmentation": study_aug,
            "optimization": study_opt
        }

In [69]:
# list all the best parameters from each stage
param_summary = {
    "window": study_window.best_trial.user_attrs["window_params"],
    "architecture": study_arch.best_trial.user_attrs["arch_params"],
    "contrastive": study_contrastive.best_trial.user_attrs["contrastive_params"],
    "augmentation": study_aug.best_trial.user_attrs["aug_params"],
    "optimization": study_opt.best_trial.user_attrs["opt_params"]
}

In [70]:
param_summary

{'window': {'window_size': 180},
 'architecture': {'hidden_dim': 256, 'num_layers': 4},
 'contrastive': {'temperature': 0.10901392823581604, 'exclusion_radius': 18},
 'augmentation': {'mask_ratio': 0.10112014206194599,
  'jitter_sigma': 0.019884546877356246},
 'optimization': {'learning_rate': 0.0012543308801674451, 'batch_size': 64}}